# 🦜🔗 LangChain Study Notes: Streaming & Batch Processing

## 📌 Quick Reference & Overview
In production LLM applications, execution speed and user responsiveness are critical:
- **Streaming (`model.stream()`)**: Delivers tokens in real-time as they are generated by the LLM, reducing perceived latency.
- **Batch Processing (`model.batch()`)**: Executes multiple independent prompts concurrently in parallel, maximizing throughput.

---

### 🌊 Part 1: Real-Time Token Streaming (`model.stream`)

#### ⚡ Why Stream?
Instead of waiting 5–10 seconds for a full response to be completely generated before displaying it, streaming prints tokens progressively chunk-by-chunk. This creates a responsive ChatGPT-like typing effect.

#### ⚙️ How it Works:
- Calling `model.stream(prompt)` returns an **iterator** yielding `AIMessageChunk` objects.
- Each chunk contains `.text` (or `.content`), allowing live UI rendering.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]= os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-3.5-flash-lite")

In [2]:
model.invoke("Write me a 200 words paragraph on Artificial Intelligence.")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': 'Artificial Intelligence (AI) has rapidly transformed from a staple of science fiction into a foundational pillar of modern civilization. At its core, AI refers to the simulation of human intelligence in machines programmed to think, learn, adapt, and problem-solve. By processing vast oceans of data at unprecedented speeds, machine learning algorithms and neural networks can recognize complex patterns, make autonomous decisions, and continuously improve their own performance without explicit human intervention. Today, AI’s footprint is ubiquitous. It powers the recommendation engines streaming our favorite movies, optimizes global supply chains, diagnoses diseases with remarkable medical precision, and drives the development of autonomous vehicles. Generative AI tools can now compose symphonies, draft code, and generate photorealistic art, fundamentally redefining the boundaries of human creativity and productivity. However, this technologica

In [4]:
for chunk in model.stream("Write me a 200 words paragraph on Artificial Intelligence."):
    print(chunk.text, end="|", flush=True)

Artificial Intelligence| (AI) has rapidly transformed from a staple of science fiction into a foundational pillar of modern civilization.| At its core, AI refers to the simulation of human intelligence in machines programmed to think, learn, reason, and solve| complex problems. By processing vast oceans of data at unprecedented speeds, machine learning algorithms and neural networks can recognize patterns, make| predictions, and continuously improve their performance without explicit human intervention. Today, AI permeates nearly every facet of our daily lives. In| healthcare, it assists in early disease detection and accelerates drug discovery. In finance, it detects fraudulent transactions in milliseconds. Furthermore|, virtual assistants, recommendation engines, and autonomous vehicles are no longer futuristic concepts, but standard conveniences. However, this technological| leap is not without its challenges. The rapid integration of AI raises critical ethical questions regarding d

In [6]:
model.invoke("Why do parrots have colorful feathers?")

AIMessage(content=[{'type': 'text', 'text': 'Parrots have colorful feathers primarily for **survival, communication, and evolution**. While to human eyes their bright greens, reds, blues, and yellows might seem like they would easily spot predators, in their natural habitats, these colors actually serve very important purposes. \n\nHere is a breakdown of why parrots are so colorful:\n\n### 1. Camouflage in the Rainforest\nIt sounds counterintuitive, but a bright green parrot is actually very well-camouflaged in a lush, green tropical rainforest canopy. The dappled sunlight filtering through the leaves creates patches of bright light and deep shadows. A parrot\'s bright green and yellow plumage helps break up its silhouette, making it difficult for predators (like hawks and eagles) to spot them against the foliage. \n\n### 2. Finding a Mate (Sexual Selection)\nFor species that aren\'t green (like scarlet macaws or sun conures), vibrant colors play a massive role in choosing a mate. Brig

In [5]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

Parrots| have colorful feathers primarily for **survival, communication, and evolution**. While to our| eyes their bright greens, reds, blues, and yellows look flashy, in their natural jungle habitats, these colors actually serve important| purposes. 

Here is a breakdown of why parrots are so colorful:

### 1. Camouflage in the Rainforest|
It might seem like a bright red or green bird would stand out, but in a lush tropical rainforest, parrot colors act| as surprisingly effective camouflage:
* **Green Parrots:** Most parrots (like many Amazons and parakeets) are predominantly| green. In the dense, sun-dappled canopy of a rainforest, green feathers help them blend seamlessly into the foliage|, hiding them from predators like hawks and eagles.
* **Flashes of Color:** Other colors, like the blue| on a Macaw's tail or the yellow on its neck, often break up the bird's silhouette, making it harder| for predators to recognize them as a distinct shape.

### 2. Finding a Mate (Sexual Selection

---
### 📦 Part 2: Parallel Batch Processing (`model.batch`)

#### ⚡ Why Batch?
When processing multiple prompts (e.g. 100 customer support tickets or document summaries), calling `.invoke()` sequentially in a `for` loop takes `N * time_per_request` seconds. `.batch()` executes requests **in parallel** under the hood, finishing in `~1 * time_per_request` seconds.

#### ⚙️ Concurrency Control (`config={"max_concurrency": N}`):
- Prevents hitting LLM provider rate-limits (HTTP 429 errors) by capping maximum simultaneous API calls.

In [7]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

content=[{'type': 'text', 'text': "Parrots have colorful feathers primarily for **survival, communication, and evolution**. While humans see their bright reds, blues, greens, and yellows as dazzling, in the wild, these colors serve very practical purposes. \n\nHere are the main reasons why parrots are so colorful:\n\n### 1. Camouflage in the Rainforest\nIt might seem strange that a bright green or red bird is camouflaged, but in their natural habitat—tropical rainforests—it makes a lot of sense. \n* **Green** parrots blend in seamlessly with the lush canopy of leaves, hiding them from predators like hawks and eagles.\n* **Flashes of bright color** (like red or blue) often break up the bird’s silhouette when they are sitting still, making it harder for predators to recognize them as a distinct shape among the dappled sunlight and shadows of the jungle.\n\n### 2. Finding a Mate (Sexual Selection)\nIn the bird world, coloration is a major factor in choosing a partner. Brighter, more vibra

In [ ]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
],
config={
    "max_concurrency": 5, # Limit to 5 parallel calls
}
)
for response in responses:
    print(response)